# Load Packages

In [1]:
import pandas as pd
import numpy as np
import pickle as pickle
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier

# Import Data

In [2]:
train_set = pd.read_csv("../data/clean/twitter_training_clean.csv", index_col=0)
test_set = pd.read_csv("../data/clean/twitter_test_clean.csv", index_col=0)
val_set = pd.read_csv("../data/clean/twitter_validation_clean.csv", index_col=0)

X_train = train_set.drop(labels="Sentiment", axis=1)
y_train = train_set["Sentiment"]

X_test = test_set.drop(labels="Sentiment", axis=1)
y_test = test_set["Sentiment"]

X_val = val_set.drop(labels="Sentiment", axis=1)
y_val = val_set["Sentiment"]

## Encoding

In [3]:
encoder = OrdinalEncoder()
encoder = encoder.fit(X_train)
X_train = encoder.transform(X_train)

encoder = encoder.fit(X_val)
X_val = encoder.transform(X_val)

encoder = encoder.fit(X_test)
X_test = encoder.transform(X_test)

## Scaling

In [4]:
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)

# Model

In [5]:
adaboost = AdaBoostClassifier(random_state=42)
matthews_scorer = make_scorer(matthews_corrcoef)

## Hyperparameter Search

In [6]:
param_grid = {
    "n_estimators": [10, 20, 50, 100, 200, 500, 1000],
    "learning_rate": [0.1, 0.2, 0.5, 0.7, 0.9, 1.0]
}

grid_search = GridSearchCV(adaboost, param_grid, scoring=matthews_scorer, verbose=3)
grid_search.fit(X_train, y_train)

grid_search.cv_results_

Fitting 5 folds for each of 42 candidates, totalling 210 fits
[CV 1/5] END learning_rate=0.1, n_estimators=10;, score=0.125 total time=   2.2s
[CV 2/5] END learning_rate=0.1, n_estimators=10;, score=0.102 total time=   1.8s
[CV 3/5] END learning_rate=0.1, n_estimators=10;, score=0.110 total time=   2.1s
[CV 4/5] END learning_rate=0.1, n_estimators=10;, score=0.102 total time=   2.1s
[CV 5/5] END learning_rate=0.1, n_estimators=10;, score=0.120 total time=   2.1s
[CV 1/5] END learning_rate=0.1, n_estimators=20;, score=0.057 total time=   4.0s
[CV 2/5] END learning_rate=0.1, n_estimators=20;, score=0.046 total time=   4.1s
[CV 3/5] END learning_rate=0.1, n_estimators=20;, score=0.053 total time=   4.1s
[CV 4/5] END learning_rate=0.1, n_estimators=20;, score=0.044 total time=   4.1s
[CV 5/5] END learning_rate=0.1, n_estimators=20;, score=0.057 total time=   4.1s
[CV 1/5] END learning_rate=0.1, n_estimators=50;, score=0.076 total time=  10.3s
[CV 2/5] END learning_rate=0.1, n_estimators=50

{'mean_fit_time': array([2.04717751e+00, 4.02923512e+00, 7.74777040e+00, 7.52987700e+00,
        1.54168405e+01, 3.91017149e+01, 7.73525898e+01, 7.89894438e-01,
        1.65326233e+00, 3.85202579e+00, 7.77369413e+00, 1.51336299e+01,
        3.91230578e+01, 7.72384991e+01, 8.02392817e-01, 1.58804679e+00,
        3.92989836e+00, 7.73214693e+00, 1.52907035e+01, 3.88584112e+01,
        9.86662495e+03, 1.76079965e+00, 3.70750480e+00, 9.01074057e+00,
        1.84841195e+01, 3.68465833e+01, 9.36009254e+01, 1.08204386e+02,
        1.35867500e+00, 2.06564541e+00, 3.94362960e+00, 7.41627827e+00,
        1.43682222e+01, 3.14396952e+01, 6.24819053e+01, 6.24122477e-01,
        1.22649794e+00, 2.98967605e+00, 6.23086863e+00, 1.20512771e+01,
        3.01467929e+01, 5.99565488e+01]),
 'std_fit_time': array([1.33013527e-01, 4.60501218e-02, 2.63268341e+00, 3.63591038e-01,
        2.63629714e-01, 8.35245269e-01, 1.01031937e+00, 5.07453350e-02,
        6.32946618e-02, 1.01015394e-01, 2.16772067e-01, 2.617

In [7]:
best_model = grid_search.best_estimator_

## Training

In [8]:
train_score = best_model.score(X_train, y_train)
print("Training Score: ", train_score)

Training Score:  0.4282890735860531


## Validation

In [9]:
val_score = best_model.score(X_val, y_val)
print("Validation Score: ", val_score)

Validation Score:  0.4160810810810811


## Test

In [10]:
test_score = best_model.score(X_test, y_test)
print("Test Score: ", test_score)

Test Score:  0.176
